In [ ]:
from scipy.special import sph_harm, genlaguerre, factorial

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.special import sph_harm_y, genlaguerre, factorial

def R_nl(r, n, l, a0=1.0):
    """
    Hydrogen radial wavefunction R_{nl}(r) in atomic units (a0=1 by default).
    Normalized so that ∫ |R_{nl}|^2 r^2 dr = 1.
    """
    rho = 2.0 * r / (n * a0)

    # Associated Laguerre polynomial L_{n-l-1}^{2l+1}(rho)
    L = genlaguerre(n - l - 1, 2*l + 1)(rho)

    # Normalization constant
    pref = (2.0/(n*a0))**3
    norm = np.sqrt(pref * factorial(n - l - 1) / (2*n * factorial(n + l)))

    return norm * np.exp(-rho/2) * rho**l * L


def Y_lm(theta, phi, l, m):
    """
    Spherical harmonic Y_l^m(theta, phi).
    scipy.special.sph_harm uses arguments (m, l, phi, theta).
    """
    return sph_harm_y(m, l, phi, theta)

def psi_nlm(r, theta, phi, n, l, m, a0=1.0):
    """Full spatial wavefunction ψ_{nlm}(r,θ,φ)."""
    return R_nl(r, n, l, a0=a0) * Y_lm(theta, phi, l, m)


def density_nlm(r, theta, phi, n, l, m, a0=1.0):
    """
    Energy/probability density u ∝ |ψ|^2 (normalized probability density).
    In the LC/Q lens: treat this as "stored reactive energy density pattern."
    """
    psi = psi_nlm(r, theta, phi, n, l, m, a0=a0)
    return np.abs(psi)**2


In [ ]:
def cart_to_sph(x, y, z):
    r = np.sqrt(x*x + y*y + z*z)
    # theta: polar angle from +z (0..pi)
    theta = np.arccos(np.clip(z / np.where(r == 0, 1.0, r), -1.0, 1.0))
    # phi: azimuth angle in x-y plane (0..2pi)
    phi = np.mod(np.arctan2(y, x), 2*np.pi)
    return r, theta, phi


In [ ]:
def plot_density_slice(n, l, m, extent=20.0, N=600, plane="xz", a0=1.0, log_scale=True):
    """
    Plot |ψ|^2 on a 2D slice.
    extent: axis range in units of a0 (Bohr radii).
    plane: "xz", "xy", or "yz"
    log_scale: log10 plot helps reveal structure over large dynamic range.
    """
    grid = np.linspace(-extent, extent, N)
    A, B = np.meshgrid(grid, grid, indexing="xy")

    if plane == "xz":
        x, y, z = A, 0*A, B
        xlabel, ylabel = "x / a0", "z / a0"
    elif plane == "xy":
        x, y, z = A, B, 0*A
        xlabel, ylabel = "x / a0", "y / a0"
    elif plane == "yz":
        x, y, z = 0*A, A, B
        xlabel, ylabel = "y / a0", "z / a0"
    else:
        raise ValueError("plane must be one of: 'xz', 'xy', 'yz'")

    r, theta, phi = cart_to_sph(x, y, z)

    dens = density_nlm(r, theta, phi, n, l, m, a0=a0)

    # avoid log(0)
    eps = 1e-20
    show = np.log10(dens + eps) if log_scale else dens

    plt.figure(figsize=(7, 6))
    im = plt.imshow(
        show,
        origin="lower",
        extent=[-extent, extent, -extent, extent],
        aspect="equal",
    )
    plt.colorbar(im, label=("log10 |ψ|^2" if log_scale else "|ψ|^2"))
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(f"Hydrogen density slice: n={n}, l={l}, m={m} (plane {plane})")
    plt.tight_layout()
    plt.show()


In [ ]:
# 1s
plot_density_slice(n=1, l=0, m=0, extent=15, plane="xz")

# 2p (choose m=0 for the classic dumbbell along z)
plot_density_slice(n=2, l=1, m=0, extent=25, plane="xz")

# 2p (m=±1 gives different angular structure)
plot_density_slice(n=2, l=1, m=1, extent=25, plane="xz")

# 3d (classic clover-like patterns show up strongly for l=2)
plot_density_slice(n=3, l=2, m=0, extent=35, plane="xz")
plot_density_slice(n=3, l=2, m=2, extent=35, plane="xy")
